# From Linguistic Context to Semantic Structure — Tests

Or Shulrufer

**Purpose — test notebook.** This optional notebook checks the actual computational notebook. It reads that notebook's source cells and exercises their functions without copying their implementation. It uses explicitly constructed unit-test inputs, which are not corpus samples or classification evidence. The computational experiment runs in GitHub Actions.

The current tests cover the implemented acquisition and normalization stage. This notebook will gain stage-specific tests as chunking, numerical operations and classification are connected to the same computational notebook. Passing these tests does not establish full experiment completion or classification performance.

The figures notebook defines manuscript illustrations without rendering before approval. Unicode Markdown and LaTeX are the two manuscript sources. After all notebooks are validated, an optional single Python pipeline will be derived for deployment through suitable scikit-learn and PyTorch interfaces.

In [ ]:
# Purpose: use the computational notebook's one dependency specification.
from pathlib import Path
import json
project = Path.cwd().resolve()
if (project / "Paper_Final").is_dir():
    project = project / "Paper_Final"
computational_notebook = json.loads((project / "Proof.ipynb").read_text())
dependency_cells = [cell for cell in computational_notebook["cells"] if cell.get("id") == "dependencies"]
assert len(dependency_cells) == 1
exec(compile("".join(dependency_cells[0]["source"]), "computational-notebook-dependencies", "exec"))


In [ ]:
# Purpose: test the canonical computational notebook without maintaining a second implementation.
from pathlib import Path
import ast
import hashlib
import json
import tempfile
import unittest
from unittest.mock import patch
import io
import math
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime, format_datetime
from threading import Lock
from collections import Counter
import unicodedata
import urllib.request
import urllib.error
import time
import nbformat

PROJECT_DIRECTORY = Path.cwd().resolve()
if (PROJECT_DIRECTORY / "Paper_Final").is_dir():
    PROJECT_DIRECTORY = PROJECT_DIRECTORY / "Paper_Final"
COMPUTATION_PATH = PROJECT_DIRECTORY / "Proof.ipynb"
FIGURES_PATH = PROJECT_DIRECTORY / "Figure_Generator.ipynb"
computation = nbformat.read(COMPUTATION_PATH, as_version=4)
figures = nbformat.read(FIGURES_PATH, as_version=4)

def actual_function(notebook, name, namespace):
    matches = []
    for cell in notebook.cells:
        if cell.cell_type == "code":
            matches.extend(node for node in ast.parse(cell.source).body
                           if isinstance(node, ast.FunctionDef) and node.name == name)
    assert len(matches) == 1, f"Expected one canonical definition of {name}"
    exec(compile(ast.Module(body=matches, type_ignores=[]), str(COMPUTATION_PATH), "exec"), namespace)
    return namespace[name]

namespace = {"Counter": Counter, "unicodedata": unicodedata, "hashlib": hashlib,
             "urllib": urllib, "time": time, "json": json,
             "math": math, "datetime": datetime, "timezone": timezone,
             "parsedate_to_datetime": parsedate_to_datetime,
             "SOURCE_REQUEST_LOCK": Lock(), "NEXT_SOURCE_REQUEST_AT": [0.0],
             "SOURCE_REQUEST_INTERVAL": 0.2}
normalize_text = actual_function(computation, "normalize_text", namespace)
verify_source_bytes = actual_function(computation, "verify_source_bytes", namespace)
wait_for_source_request = actual_function(computation, "wait_for_source_request", namespace)
defer_source_requests = actual_function(computation, "defer_source_requests", namespace)
fetch_source_bytes = actual_function(computation, "fetch_source_bytes", namespace)
obtain_existing_text = actual_function(computation, "obtain_existing_text", namespace)

class ComputationalNotebookTests(unittest.TestCase):
    def test_notebooks_have_valid_unique_cells(self):
        for notebook in (computation, figures):
            nbformat.validate(notebook)
            self.assertEqual(len({cell.id for cell in notebook.cells}), len(notebook.cells))
            for index, cell in enumerate(notebook.cells):
                if cell.cell_type == "code":
                    compile(cell.source, f"notebook-cell-{index}", "exec")

    def test_character_normalization(self):
        text, edits, counts = normalize_text("Hello\r\nWORLD\tA\u00a0B\x00")
        self.assertEqual(text, "hello\nworld a b")
        self.assertTrue(edits)
        self.assertEqual(counts["nonprinting_removed"], 1)
        self.assertEqual(normalize_text(text)[0], text)

    def test_unicode_lowercase_expansion_is_recorded(self):
        text, edits, counts = normalize_text("İ")
        self.assertEqual(text, "i\u0307")
        self.assertEqual(edits[0]["source_end"], 1)
        self.assertEqual(edits[0]["normalized_end"], 2)
        self.assertEqual(counts["lowercase_character_expansion"], 1)

    def test_empty_broken_and_long_texts_remain(self):
        self.assertEqual(normalize_text("")[0], "")
        self.assertEqual(normalize_text("Words WITHOUT sentence punctuation")[0], "words without sentence punctuation")
        raw = "Mixed text without a final sentence. " * 100000
        self.assertEqual(normalize_text(raw)[0], raw.lower())
        self.assertEqual(len(normalize_text(raw)[0]), len(raw))

    def test_original_bytes_require_complete_length_and_hash(self):
        data = b"Existing extracted text\n"
        asset = {"path": "2001.00001/full.md", "bytes": len(data), "sha256": None,
                 "git_blob_sha1": hashlib.sha1(f"blob {len(data)}\0".encode() + data).hexdigest()}
        verify_source_bytes(data, asset)
        with self.assertRaises(AssertionError):
            verify_source_bytes(data[:-1], asset)
        with self.assertRaises(AssertionError):
            verify_source_bytes(b"X" + data[1:], asset)
        asset["sha256"] = hashlib.sha256(data).hexdigest()
        verify_source_bytes(data, asset)

    def test_resume_reuses_verified_bytes_without_network(self):
        data = b"Already downloaded complete text"
        asset = {"path": "2001.00001/full.md", "bytes": len(data),
                 "sha256": hashlib.sha256(data).hexdigest(), "git_blob_sha1": None}
        record = {"document_id": "2001.00001", "assets": [asset]}
        with tempfile.TemporaryDirectory() as directory:
            directory = Path(directory)
            (directory / "2001.00001.md").write_bytes(data)
            namespace["RAW_TEXT_DIRECTORY"] = directory
            with patch("urllib.request.urlopen", side_effect=AssertionError("Resume unexpectedly used the network")):
                result = obtain_existing_text(record)
            self.assertEqual(result["bytes"], len(data))
            self.assertEqual((directory / "2001.00001.md").read_bytes(), data)

    def test_shared_request_schedule_spaces_requests(self):
        clock = [100.0]
        namespace["NEXT_SOURCE_REQUEST_AT"] = [100.0]
        with patch("time.monotonic", side_effect=lambda: clock[0]), patch("time.sleep", side_effect=lambda delay: clock.__setitem__(0, clock[0] + delay)):
            wait_for_source_request()
            wait_for_source_request()
            wait_for_source_request()
        self.assertAlmostEqual(clock[0], 100.4)

    def test_rate_limit_pauses_all_workers_and_honors_retry_after(self):
        namespace["NEXT_SOURCE_REQUEST_AT"] = [0.0]
        with patch("time.monotonic", return_value=100.0):
            delay = defer_source_requests({"Retry-After": "120"}, 0, 429)
            self.assertEqual(delay, 120.0)
            self.assertEqual(namespace["NEXT_SOURCE_REQUEST_AT"][0], 220.0)
            defer_source_requests({}, 0, 503)
            self.assertEqual(namespace["NEXT_SOURCE_REQUEST_AT"][0], 220.0)

    def test_retry_after_http_date_and_malformed_header(self):
        future = datetime.now(timezone.utc) + timedelta(seconds=120)
        delay = defer_source_requests({"Retry-After": format_datetime(future, usegmt=True)}, 0, 429)
        self.assertGreater(delay, 115)
        self.assertLessEqual(delay, 120)
        self.assertEqual(defer_source_requests({"Retry-After": "invalid"}, 0, 429), 60.0)

    def test_rate_limited_download_retries_and_returns_complete_bytes(self):
        error = urllib.error.HTTPError("https://example.invalid", 429, "Too Many Requests", {"Retry-After": "120"}, None)
        response = io.BytesIO(b"complete source bytes")
        with patch.dict(namespace, {"wait_for_source_request": lambda: None}), patch("urllib.request.urlopen", side_effect=[error, response]) as opened:
            with patch.dict(namespace, {"defer_source_requests": lambda headers, attempt, status: self.assertEqual((headers["Retry-After"], attempt, status), ("120", 0, 429))}):
                self.assertEqual(fetch_source_bytes("https://example.invalid"), b"complete source bytes")
            self.assertEqual(opened.call_count, 2)

    def test_permanent_http_error_is_not_retried(self):
        error = urllib.error.HTTPError("https://example.invalid", 404, "Not Found", {}, None)
        with patch.dict(namespace, {"wait_for_source_request": lambda: None}), patch("urllib.request.urlopen", side_effect=error) as opened:
            with self.assertRaises(urllib.error.HTTPError):
                fetch_source_bytes("https://example.invalid")
            self.assertEqual(opened.call_count, 1)

    def test_repeated_rate_limit_preserves_failure_for_resumable_run(self):
        error = urllib.error.HTTPError("https://example.invalid", 429, "Too Many Requests", {}, None)
        with patch.dict(namespace, {"wait_for_source_request": lambda: None, "defer_source_requests": lambda *args: None}), patch("urllib.request.urlopen", side_effect=error) as opened:
            with self.assertRaises(urllib.error.HTTPError):
                fetch_source_bytes("https://example.invalid")
            self.assertEqual(opened.call_count, 8)

    def test_figure_rendering_requires_approval(self):
        scope = {"FIGURES_APPROVED": False}
        guard = actual_function(figures, "require_approval", scope)
        with self.assertRaises(RuntimeError):
            guard()
        for cell in figures.cells:
            if cell.cell_type == "code":
                for node in ast.parse(cell.source).body:
                    if isinstance(node, ast.Assign):
                        if any(isinstance(t, ast.Name) and t.id == "RENDER_FIGURES" for t in node.targets):
                            self.assertIs(ast.literal_eval(node.value), False)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ComputationalNotebookTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Revise the computational notebook and rerun the affected tests."
print(json.dumps({"scope": "Implemented acquisition and normalization functions; figure-rendering guard",
                  "tests_run": test_result.testsRun, "successful": test_result.wasSuccessful(),
                  "experiment_complete": False,
                  "computational_notebook_sha256": hashlib.sha256(COMPUTATION_PATH.read_bytes()).hexdigest()}, indent=2))
